# 06 - Performance Tuning: Liquid Clustering + OPTIMIZE
The real fact table (112K rows) is too small to show measurable gains, so it is scaled 30x (~3.4M rows) for benchmarking.

| Scenario | Setup |
|---|---|
| **Before** | 400 small, unsorted files (simulates the small-file problem) |
| **After** | Liquid clustering on `purchase_date_key, product_sk` + `OPTIMIZE` |

**Test query:** revenue by category for January 2018, a typical dashboard filter.
**Metrics:** total files, files containing matching rows (data skipping), median query time over 5 runs.

In [0]:
from pyspark.sql import functions as F

fact = spark.table("retail_lakehouse.gold.fact_orders")

# Scale 30x so performance differences are measurable
scaled = (
    fact.crossJoin(spark.range(30).withColumnRenamed("id", "copy_id"))
        .withColumn("order_id", F.concat_ws("-", "order_id", "copy_id"))
        .drop("copy_id")
)

# repartition(400) shuffles rows randomly -> 400 small files, every file has every date mixed in
(
    scaled.repartition(400)
          .write.mode("overwrite")
          .option("overwriteSchema", "true")
          .saveAsTable("retail_lakehouse.gold.perf_fact_baseline")
)

print(f"Rows: {spark.table('retail_lakehouse.gold.perf_fact_baseline').count():,}")

In [0]:
%sql
CREATE OR REPLACE TABLE retail_lakehouse.gold.perf_fact_clustered
CLUSTER BY (purchase_date_key, product_sk)
TBLPROPERTIES ('delta.targetFileSize' = '16mb')
COMMENT 'Benchmark copy of fact_orders with liquid clustering'
AS SELECT * FROM retail_lakehouse.gold.perf_fact_baseline;

In [0]:
%sql
OPTIMIZE retail_lakehouse.gold.perf_fact_clustered;

In [0]:
tables = {
    "before": "retail_lakehouse.gold.perf_fact_baseline",
    "after":  "retail_lakehouse.gold.perf_fact_clustered",
}

rows = []
for scenario, table in tables.items():
    d = spark.sql(f"DESCRIBE DETAIL {table}").first()
    rows.append((scenario, int(d.numFiles), round(d.sizeInBytes / 1024 / 1024, 1), str(d.clusteringColumns)))

display(spark.createDataFrame(rows, "scenario STRING, num_files INT, size_mb DOUBLE, clustering_columns STRING"))

In [0]:
import time
import statistics
from pyspark.sql import functions as F

QUERY = """
SELECT p.product_category_name,
       COUNT(*)          AS items_sold,
       SUM(f.item_total) AS revenue
FROM {table} f
JOIN retail_lakehouse.gold.dim_product p ON f.product_sk = p.product_sk
WHERE f.purchase_date_key BETWEEN 20180101 AND 20180131
GROUP BY p.product_category_name
ORDER BY revenue DESC
"""

def median_query_seconds(table, runs=5):
    spark.sql(QUERY.format(table=table)).collect()        # warm-up run (not counted)
    times = []
    for _ in range(runs):
        start = time.perf_counter()
        spark.sql(QUERY.format(table=table)).collect()
        times.append(time.perf_counter() - start)
    return round(statistics.median(times), 3)

def files_with_matching_rows(table):
    """How many files actually contain January 2018 rows (lower = better data skipping)."""
    return spark.sql(f"""
        SELECT COUNT(DISTINCT _metadata.file_path)
        FROM {table}
        WHERE purchase_date_key BETWEEN 20180101 AND 20180131
    """).first()[0]

scenarios = [
    ("Before: 400 small unsorted files",     "retail_lakehouse.gold.perf_fact_baseline"),
    ("After: liquid clustering + OPTIMIZE",  "retail_lakehouse.gold.perf_fact_clustered"),
]

rows = []
for label, table in scenarios:
    detail = spark.sql(f"DESCRIBE DETAIL {table}").first()
    rows.append((
        label,
        table.split(".")[-1],
        int(detail.numFiles),
        round(detail.sizeInBytes / 1024 / 1024, 1),
        int(files_with_matching_rows(table)),
        float(median_query_seconds(table)),
    ))
    print(f"✅ measured {label}")

results = spark.createDataFrame(
    rows,
    "scenario STRING, table_name STRING, total_files INT, size_mb DOUBLE, "
    "files_with_jan_2018_rows INT, median_query_seconds DOUBLE",
).withColumn("measured_at", F.current_timestamp())

results.write.mode("append").saveAsTable("retail_lakehouse.gold.perf_benchmark_results")
display(results)

In [0]:
%sql
SELECT SUM(item_total) AS jan_2018_revenue
FROM retail_lakehouse.gold.perf_fact_baseline
WHERE purchase_date_key BETWEEN 20180101 AND 20180131;

In [0]:
%sql
DESCRIBE DETAIL retail_lakehouse.gold.fact_orders;

## Cleanup
Drops the 3.4M-row benchmark tables after measuring. Results are kept in `gold.perf_benchmark_results`.
To re-run the benchmark, run the notebook from Cell 2.

In [0]:
%sql
DROP TABLE IF EXISTS retail_lakehouse.gold.perf_fact_baseline;
DROP TABLE IF EXISTS retail_lakehouse.gold.perf_fact_clustered;
-- perf_benchmark_results stays as your record